# ⚡ Power System Faults — CEO/CTO Executive Dashboard, built by Claude

**Goal.** Read the *Power System Faults* dataset, profile it, and ask **Claude** (Anthropic API) to design a
fully interactive, leadership-level dashboard delivered as **one self-contained HTML file**.

**Pipeline**

| # | Step | What happens |
|---|------|--------------|
| 1 | Setup | Install / import libraries |
| 2 | Credentials | Load `ANTHROPIC_API_KEY` from a local `.env` file |
| 3 | Data | Download the CSV, clean it, split the lat/long column |
| 4 | Profiling | Compute KPIs and breakdowns that become Claude's *grounding facts* |
| 5 | Prompt engineering | System prompt (role + design rules) and user prompt (schema, facts, chart spec) |
| 6 | Generation | Stream the dashboard HTML from Claude |
| 7 | Post-processing | Inject the full dataset, optionally inline Plotly so the file works offline, validate |
| 8 | Delivery | Save `power_faults_dashboard.html`, preview it inline, give a download link |

> **Design choice — data injection.** Claude writes the dashboard *code* with a placeholder
> (`__DATA_JSON__`) instead of retyping 500+ rows. Python then injects the real records. This keeps the
> response short and cheap, and guarantees the numbers in the dashboard are exactly the dataset's numbers
> (the model can't mistype or invent a value).

## 1. Setup — install and import libraries

In [1]:
# Run once per environment (uncomment if the packages are missing or outdated), then restart the kernel.
# -U upgrades `anthropic` so newer API options (output_config, fallbacks) are supported natively.
# %pip install -q -U anthropic python-dotenv pandas requests

In [2]:
import os, re, json, html
from pathlib import Path

import pandas as pd
import requests
import anthropic                       # official Anthropic SDK
from dotenv import load_dotenv         # reads KEY=VALUE pairs from a .env file
from IPython.display import IFrame, FileLink, Markdown, display

print("anthropic SDK version:", anthropic.__version__)

anthropic SDK version: 1.13.0


## 2. Load the Claude API key from `.env`

Create a file called `.env` **next to this notebook** containing:

```
ANTHROPIC_API_KEY=sk-ant-...
```

Never hard-code the key in the notebook and never commit `.env` to git.

In [3]:
# load_dotenv() searches the current folder (and its parents) for .env and copies its
# variables into os.environ. override=False keeps any key already exported in the shell.
load_dotenv(override=False)

API_KEY = os.getenv("ANTHROPIC_API_KEY")
if not API_KEY:
    raise RuntimeError(
        "ANTHROPIC_API_KEY not found. Add a line `ANTHROPIC_API_KEY=sk-ant-...` "
        "to the .env file next to this notebook, then re-run this cell."
    )

# Show only a masked version so the key never ends up in saved notebook output.
print(f"Key loaded: {API_KEY[:7]}…{API_KEY[-4:]}  (length {len(API_KEY)})")

client = anthropic.Anthropic(api_key=API_KEY, max_retries=3)

Key loaded: sk-ant-…8wAA  (length 106)


## 3. Load and clean the dataset

In [4]:
DATA_URL = "https://raw.githubusercontent.com/Dr-AlaaKhamis/ISE518/refs/heads/main/datasets/fault_data.csv"

raw = pd.read_csv(DATA_URL)
print("Shape:", raw.shape)
raw.head()

Shape: (506, 13)


,Fault ID,Fault Type,"Fault Location (Latitude, Longitude)",Voltage (V),Current (A),Power Load (MW),Temperature (C),Wind Speed (km/h),Weather Condition,Maintenance Status,Component Health,Duration of Fault (hrs),Down time (hrs)
0,F001,Line Breakage,"(34.0522, -118.2437)",2200,250,50,25,20,Clear,Scheduled,Normal,2.0,1.0
1,F002,Transformer Failure,"(34.056, -118.245)",1800,180,45,28,15,Rainy,Completed,Faulty,3.0,5.0
2,F003,Overheating,"(34.0525, -118.244)",2100,230,55,35,25,Windstorm,Pending,Overheated,4.0,6.0
3,F004,Line Breakage,"(34.055, -118.242)",2050,240,48,23,10,Clear,Completed,Normal,2.5,3.0
4,F005,Transformer Failure,"(34.0545, -118.243)",1900,190,50,30,18,Snowy,Scheduled,Faulty,3.5,4.0


In [5]:
df = raw.copy()

# 1) Short, code-friendly column names (these become the JSON keys used by the dashboard JS).
df = df.rename(columns={
    "Fault ID": "fault_id",
    "Fault Type": "fault_type",
    "Fault Location (Latitude, Longitude)": "location",
    "Voltage (V)": "voltage_v",
    "Current (A)": "current_a",
    "Power Load (MW)": "load_mw",
    "Temperature (C)": "temp_c",
    "Wind Speed (km/h)": "wind_kmh",
    "Weather Condition": "weather",
    "Maintenance Status": "maintenance",
    "Component Health": "health",
    "Duration of Fault (hrs)": "duration_h",
    "Down time (hrs)": "downtime_h",
})

# 2) The location is stored as the text "(34.0522, -118.2437)" -> split into numeric lat / lon.
coords = df["location"].str.strip("() ").str.split(",", expand=True)
df["lat"] = pd.to_numeric(coords[0], errors="coerce")
df["lon"] = pd.to_numeric(coords[1], errors="coerce")

# 3) Force numeric types (anything unparsable becomes NaN so we can see it).
num_cols = ["voltage_v", "current_a", "load_mw", "temp_c", "wind_kmh", "duration_h", "downtime_h", "lat", "lon"]
df[num_cols] = df[num_cols].apply(pd.to_numeric, errors="coerce")

# 4) Tidy text categories (stray spaces would otherwise create duplicate slicer values).
cat_cols = ["fault_type", "weather", "maintenance", "health"]
for c in cat_cols:
    df[c] = df[c].astype(str).str.strip()

# 5) The dataset has no timestamp. Fault IDs (F001, F002, …) are sequential, so we use the
#    numeric part as an *event sequence* — the x-axis for the trend (line) chart.
df["seq"] = df["fault_id"].str.extract(r"(\d+)", expand=False).astype(int)
df = df.sort_values("seq").drop(columns="location").reset_index(drop=True)

# 6) Data-quality report.
print("Missing values per column:\n", df.isna().sum()[df.isna().sum() > 0].to_string() or "  none")
print("Duplicate fault IDs:", df["fault_id"].duplicated().sum())
df = df.dropna(subset=["lat", "lon", "downtime_h"])
print("Clean shape:", df.shape)
df.head()

Missing values per column:
 Series([], )
Duplicate fault IDs: 0
Clean shape: (506, 15)


,fault_id,fault_type,voltage_v,current_a,load_mw,temp_c,wind_kmh,weather,maintenance,health,duration_h,downtime_h,lat,lon,seq
0,F001,Line Breakage,2200,250,50,25,20,Clear,Scheduled,Normal,2.0,1.0,34.0522,-118.2437,1
1,F002,Transformer Failure,1800,180,45,28,15,Rainy,Completed,Faulty,3.0,5.0,34.0560,-118.2450,2
2,F003,Overheating,2100,230,55,35,25,Windstorm,Pending,Overheated,4.0,6.0,34.0525,-118.2440,3
3,F004,Line Breakage,2050,240,48,23,10,Clear,Completed,Normal,2.5,3.0,34.0550,-118.2420,4
4,F005,Transformer Failure,1900,190,50,30,18,Snowy,Scheduled,Faulty,3.5,4.0,34.0545,-118.2430,5


## 4. Descriptive profile — the grounding facts for Claude

An LLM designs better dashboards when it knows the **actual** shape of the data: which categories exist, how
big each one is, and where the outliers are. We compute these facts in pandas (exact) and pass them to
Claude, so any headline insight it writes is based on real numbers rather than guesses.

In [6]:
def breakdown(col):
    # Count, average downtime and total downtime per category, sorted by total downtime.
    g = df.groupby(col).agg(faults=("fault_id", "count"),
                            avg_downtime_h=("downtime_h", "mean"),
                            total_downtime_h=("downtime_h", "sum"),
                            avg_duration_h=("duration_h", "mean"))
    return g.sort_values("total_downtime_h", ascending=False).round(2)

profile = {
    "rows": int(len(df)),
    "kpis": {
        "total_faults": int(len(df)),
        "total_downtime_h": round(float(df["downtime_h"].sum()), 1),
        "mean_downtime_h": round(float(df["downtime_h"].mean()), 2),
        "mean_fault_duration_h": round(float(df["duration_h"].mean()), 2),
        "share_unhealthy_components_pct": round(100 * float((df["health"] != "Normal").mean()), 1),
        "share_maintenance_pending_pct": round(100 * float((df["maintenance"] == "Pending").mean()), 1),
    },
    "numeric_summary": df[num_cols].describe().round(2).to_dict(),
    "breakdowns": {c: breakdown(c).reset_index().to_dict(orient="records") for c in cat_cols},
    "geo_bounds": {"lat": [float(df.lat.min()), float(df.lat.max())],
                   "lon": [float(df.lon.min()), float(df.lon.max())]},
    "notes": "No timestamp column; 'seq' is the numeric part of fault_id and is the event order.",
}

for c in cat_cols:
    display(Markdown(f"**By {c}**"), breakdown(c))
print(json.dumps(profile["kpis"], indent=2))

**By fault_type**

,faults,avg_downtime_h,total_downtime_h,avg_duration_h
fault_type,,,,
Transformer Failure,171,4.06,694.5,4.09
Overheating,171,4.04,691.1,4.08
Line Breakage,164,3.89,638.3,3.85


**By weather**

,faults,avg_downtime_h,total_downtime_h,avg_duration_h
weather,,,,
Rainy,117,4.13,482.7,4.12
Clear,110,3.92,430.9,3.96
Thunderstorm,100,4.15,414.6,3.85
Snowy,92,3.95,363.1,3.99
Windstorm,87,3.82,332.6,4.12


**By maintenance**

,faults,avg_downtime_h,total_downtime_h,avg_duration_h
maintenance,,,,
Completed,173,4.02,694.7,3.99
Scheduled,170,3.95,670.9,4.05
Pending,163,4.04,658.3,3.99


**By health**

,faults,avg_downtime_h,total_downtime_h,avg_duration_h
health,,,,
Overheated,173,4.26,736.7,3.95
Normal,174,3.78,658.2,4.06
Faulty,159,3.96,629.0,4.02


{
  "total_faults": 506,
  "total_downtime_h": 2023.9,
  "mean_downtime_h": 4.0,
  "mean_fault_duration_h": 4.01,
  "share_unhealthy_components_pct": 65.6,
  "share_maintenance_pending_pct": 32.2
}


In [7]:
# Cross-tab of fault type × weather — a useful hint for the heatmap and for "risk driver" insights.
xtab = pd.crosstab(df["fault_type"], df["weather"], values=df["downtime_h"], aggfunc="mean").round(2)
profile["avg_downtime_fault_x_weather"] = xtab.fillna(0).to_dict()

# Correlation of the operating conditions with downtime — candidate "drivers" for leadership.
corr = df[["voltage_v", "current_a", "load_mw", "temp_c", "wind_kmh", "duration_h", "downtime_h"]].corr()["downtime_h"].round(3)
profile["corr_with_downtime"] = corr.drop("downtime_h").to_dict()
xtab

weather,Clear,Rainy,Snowy,Thunderstorm,Windstorm
fault_type,,,,,
Line Breakage,3.56,4.09,4.01,3.69,4.08
Overheating,4.10,4.24,3.91,3.98,3.86
Transformer Failure,4.05,4.03,3.90,4.69,3.53


## 5. Prompt engineering

Prompt structure (each part has one job):

* **System prompt** — *role* (top-tier data-viz / exec-dashboard expert), *quality bar*, and *hard
  technical constraints* (one HTML file, Plotly, placeholder for data, no invented numbers).
* **User prompt**, organised in XML-style tags so Claude can find each part:
  `<dataset_schema>`, `<profile>` (the facts from step 4), `<sample_rows>`, `<requirements>`
  (charts, slicers, KPIs, insights, layout) and `<output_format>`.

Charts requested: **line** (trend over the event sequence), **bar**, **treemap**, **geospatial map**,
**heatmap**, **donut** and **scatter** — all cross-filtered by the same slicers.

In [8]:
MODEL = "claude-opus-5-5"          # most capable default Opus model
DATA_PLACEHOLDER = "__DATA_JSON__"  # Claude must leave this token exactly once; Python replaces it
PLOTLY_CDN = "https://cdn.plot.ly/plotly-2.35.2.min.js"

SYSTEM_PROMPT = f'''You are a top-tier expert in data visualization, descriptive analytics, and executive
(CEO/CTO) dashboard design. You write production-quality, dependency-light front-end code.

Your dashboards are:
- Simple: a leader understands the state of the grid in under 10 seconds.
- Insight-first: every chart answers a business question, and its title states the takeaway.
- Interactive: global slicers cross-filter every KPI and chart; hover tooltips show exact values.
- Honest: every number is computed in JavaScript from the data at runtime. You never hard-code,
  estimate, or invent figures in the HTML.

Hard technical constraints:
1. Output ONE complete HTML document (<!DOCTYPE html> … </html>). Inline all CSS and JS.
2. The only external resource allowed is Plotly, loaded with exactly:
   <script src="{PLOTLY_CDN}"></script>
3. The data is NOT in the prompt in full. Declare it exactly like this, once:
   const RAW_DATA = {DATA_PLACEHOLDER};
   It will be replaced by a JSON array of row objects before the file is opened.
4. For the map use Plotly's `scattermap` trace with map style "carto-positron" (free CARTO tiles, no token).
   Never use "open-street-map": OSM's volunteer tile server blocks pages opened from local files.
5. Must work at 1366px laptop width and degrade to one column on phones.
'''

schema = {
    "fault_id": "string, e.g. F001",
    "seq": "int, event order (numeric part of fault_id) — use as the time-like axis",
    "fault_type": f"category: {sorted(df.fault_type.unique())}",
    "weather": f"category: {sorted(df.weather.unique())}",
    "maintenance": f"category: {sorted(df.maintenance.unique())}",
    "health": f"category (component health): {sorted(df.health.unique())}",
    "voltage_v": "number, volts", "current_a": "number, amperes", "load_mw": "number, megawatts",
    "temp_c": "number, °C", "wind_kmh": "number, km/h",
    "duration_h": "number, hours the fault lasted", "downtime_h": "number, hours of service downtime",
    "lat": "number, latitude", "lon": "number, longitude",
}

USER_PROMPT = f'''Build an interactive executive dashboard for the "Power System Faults" dataset
(electrical faults across geographic locations and weather conditions; used for reliability analysis).

<dataset_schema>
{json.dumps(schema, indent=2)}
</dataset_schema>

<profile>
Exact statistics computed in pandas on the full dataset — use them to decide what to emphasise
and how to phrase insights, but compute displayed numbers in JS from RAW_DATA.
{json.dumps(profile, indent=2, default=str)}
</profile>

<sample_rows>
{df.head(5).to_json(orient="records", indent=1)}
</sample_rows>

<requirements>
Header
- Title "Power System Reliability — Executive Fault Dashboard", subtitle with the active filter summary
  and the record count, plus a "Reset filters" button.

Slicers / filters (global; every KPI and chart re-renders on change)
- Multi-select chip/checkbox groups for Fault Type, Weather Condition, Maintenance Status, Component Health.
- A dual-handle (min/max) range filter for Downtime (hrs) and one for Event sequence (seq).
- Clicking a bar, treemap tile or donut slice toggles that category in the matching slicer (cross-filtering).

KPI cards (with small context line, e.g. vs. dataset average)
- Total faults · Total downtime (hrs) · Avg downtime per fault · Avg fault duration ·
  % components not Normal · % maintenance Pending.

Charts (at least these 7, each with a takeaway-style title and a one-line caption)
1. LINE — reliability trend: avg downtime and fault count per bucket of 25 consecutive events (seq),
   two y-axes, with a dashed overall-average reference line.
2. BAR — total downtime by fault type (horizontal, sorted, highest highlighted).
3. TREEMAP — fault count hierarchy Weather → Fault Type, tile colour = avg downtime.
4. GEOSPATIAL MAP — every fault as a point at lat/lon, size = downtime, colour = fault type,
   rich hover (id, type, weather, health, downtime). Auto-fit to the filtered points.
5. HEATMAP — avg downtime by Fault Type × Weather (annotated cells).
6. DONUT — maintenance status share, with % in centre for "Pending".
7. SCATTER — power load (MW) vs downtime, coloured by component health (risk driver view).

Key insights panel
- 3–5 auto-generated bullet insights recomputed from the filtered data in JS (e.g. top downtime driver,
  worst weather condition, share of downtime from non-Normal components, pending-maintenance exposure).
  Phrase them for executives: plain language, number first, action implied.

Design
- Clean, modern corporate look: light background, white cards with subtle shadow, 12-col CSS grid,
  one accent colour + a consistent categorical palette shared by every chart for the same category.
- Readable fonts (system UI stack), no chart junk, consistent number formatting (1 decimal for hours).
- Plotly config: responsive, displaylogo false. Handle the empty-filter case with a friendly message.
- Comment the JS so an analyst can maintain it.
</requirements>

<output_format>
Return ONLY the HTML document inside a single ```html fenced block. No explanation before or after.
</output_format>
'''

print(f"System prompt: {len(SYSTEM_PROMPT):,} chars | User prompt: {len(USER_PROMPT):,} chars")

System prompt: 1,333 chars | User prompt: 10,670 chars


## 6. Generate the dashboard with Claude

* **Streaming** is used because the response is long (a full HTML app) — it avoids HTTP timeouts.
* **Adaptive thinking + `effort="high"`** lets Claude plan the layout and JS before writing it.
* **Refusal fallback** (`fallbacks="default"`, beta) — if a safety classifier ever declines, the API
  retries on a fallback model in the same call. Set `USE_FALLBACK = False` to disable.
* We check `stop_reason` before trusting the output (`max_tokens` = truncated, `refusal` = declined).

In [9]:
USE_FALLBACK = True
MAX_TOKENS = 64000

request = dict(
    model=MODEL,
    max_tokens=MAX_TOKENS,
    system=SYSTEM_PROMPT,
    messages=[{"role": "user", "content": USER_PROMPT}],
    thinking={"type": "adaptive"},
    output_config={"effort": "high"},
)
if USE_FALLBACK:
    request.update(betas=["server-side-fallback-2026-07-01"], fallbacks="default")

# Compatibility with older `anthropic` SDK versions: parameters the installed SDK doesn't know
# yet (e.g. output_config, fallbacks) are sent as raw JSON via extra_body / extra_headers,
# which every SDK version supports. Upgrading (`%pip install -U anthropic`) makes this a no-op.
import inspect
_supported = set(inspect.signature(client.beta.messages.stream).parameters)
_extra_body = {k: request.pop(k) for k in ["thinking", "output_config", "fallbacks"]
               if k in request and k not in _supported}
if "betas" in request and "betas" not in _supported:
    request["extra_headers"] = {"anthropic-beta": ",".join(request.pop("betas"))}
if _extra_body:
    request["extra_body"] = _extra_body
    print(f"Older anthropic SDK ({anthropic.__version__}) - sending {list(_extra_body)} via extra_body.")

try:
    # Print a dot every ~2,000 characters so you can see progress during the 1–4 minute generation.
    chars = 0
    with client.beta.messages.stream(**request) as stream:
        for text in stream.text_stream:
            chars += len(text)
            if chars // 2000 > (chars - len(text)) // 2000:
                print(".", end="", flush=True)
        response = stream.get_final_message()
except anthropic.AuthenticationError:
    raise SystemExit("Invalid ANTHROPIC_API_KEY — check your .env file.")
except anthropic.RateLimitError as e:
    raise SystemExit(f"Rate limited — wait and retry. ({e.message})")
except anthropic.APIStatusError as e:
    raise SystemExit(f"API error {e.status_code}: {e.message}")
except anthropic.APIConnectionError:
    raise SystemExit("Network error — check your internet connection.")

print(f"\nstop_reason={response.stop_reason} | served by {response.model} | "
      f"input={response.usage.input_tokens:,} output={response.usage.output_tokens:,} tokens")

if response.stop_reason == "refusal":
    raise RuntimeError(f"Request declined: {getattr(response, 'stop_details', None)}")
if response.stop_reason == "max_tokens":
    raise RuntimeError("Output was truncated — raise MAX_TOKENS or lower effort and re-run.")

# Join all text blocks (thinking blocks are skipped).
answer = "".join(b.text for b in response.content if b.type == "text")

........................
stop_reason=end_turn | served by claude-opus-5-5 | input=5,462 output=47,605 tokens


## 7. Post-process: extract → inject data → (optionally) inline Plotly → validate

In [10]:
# 7a) Pull the HTML out of the ```html fence (fall back to the raw <!DOCTYPE …</html> span).
m = re.search(r"```html\s*(.*?)```", answer, re.S | re.I) or re.search(r"(<!DOCTYPE html.*</html>)", answer, re.S | re.I)
if not m:
    raise ValueError("No HTML document found in Claude's answer:\n" + answer[:500])
dashboard_html = m.group(1).strip()

# 7b) Inject the real dataset where Claude left the placeholder.
#     </ is escaped so a string in the data can never close the <script> tag early.
records_json = df.to_json(orient="records").replace("</", "<\\/")
n_placeholders = dashboard_html.count(DATA_PLACEHOLDER)
if n_placeholders != 1:
    raise ValueError(f"Expected exactly 1 '{DATA_PLACEHOLDER}' in the HTML, found {n_placeholders}.")
dashboard_html = dashboard_html.replace(DATA_PLACEHOLDER, records_json)

# 7c) Basemap fix: OpenStreetMap's volunteer tile server blocks requests without a Referer header,
#     which is the case for a local HTML file or a notebook iframe ("Access blocked … tile usage policy").
#     Swap any OSM style for the free CARTO basemap built into Plotly (no token, no Referer needed).
n_osm = len(re.findall(r"['\"]open-street-map['\"]", dashboard_html)) + dashboard_html.count("tile.openstreetmap.org")
dashboard_html = dashboard_html.replace("'open-street-map'", "'carto-positron'").replace('"open-street-map"', '"carto-positron"')
if n_osm:
    print(f"Replaced {n_osm} OpenStreetMap basemap reference(s) with carto-positron.")
if "tile.openstreetmap.org" in dashboard_html:
    print("WARNING: raw OSM tile URLs remain - set the map style to 'carto-positron' manually.")

# 7d) Make the file fully self-contained (works offline, e.g. in a boardroom) by inlining Plotly.
#     The map tiles still need internet; everything else works offline.
INLINE_PLOTLY = True
if INLINE_PLOTLY:
    tag = re.search(r'<script[^>]+src="[^"]*plotly[^"]*"[^>]*>\s*</script>', dashboard_html, re.I)
    if tag:
        plotly_js = requests.get(PLOTLY_CDN, timeout=60).text.replace("</script", "<\\/script")
        dashboard_html = dashboard_html.replace(tag.group(0), f"<script>{plotly_js}</script>")
        print(f"Inlined Plotly ({len(plotly_js)/1e6:.1f} MB).")
    else:
        print("Plotly <script src> tag not found — left as is.")

# 7e) Lightweight quality checks on what Claude produced.
checks = {
    "line chart":     bool(re.search(r"type\s*:\s*['\"]scatter['\"]", dashboard_html)) and "lines" in dashboard_html,
    "bar chart":      bool(re.search(r"type\s*:\s*['\"]bar['\"]", dashboard_html)),
    "treemap":        "treemap" in dashboard_html,
    "geospatial map": bool(re.search(r"scattermap|scattermapbox|scattergeo", dashboard_html)),
    "heatmap":        "heatmap" in dashboard_html,
    "donut/pie":      bool(re.search(r"type\s*:\s*['\"]pie['\"]", dashboard_html)),
    "slicers":        bool(re.search(r"checkbox|<select", dashboard_html, re.I)),
    "range filters":  bool(re.search(r"type\s*=\s*['\"]range", dashboard_html, re.I)),
    "reset button":   "reset" in dashboard_html.lower(),
    "data injected":  DATA_PLACEHOLDER not in dashboard_html,
}
for k, ok in checks.items():
    print("[OK]  " if ok else "[MISSING]", k)

Inlined Plotly (4.6 MB).
[OK]   line chart
[OK]   bar chart
[OK]   treemap
[OK]   geospatial map
[OK]   heatmap
[OK]   donut/pie
[MISSING] slicers
[OK]   range filters
[OK]   reset button
[OK]   data injected


## 8. Save, preview and download

In [11]:
OUT = Path("power_faults_dashboard.html")
OUT.write_text(dashboard_html, encoding="utf-8")
print(f"Saved {OUT.resolve()}  ({OUT.stat().st_size/1e6:.2f} MB)")

# Also keep the raw model answer for auditing / prompt iteration.
Path("claude_raw_response.md").write_text(answer, encoding="utf-8")

display(FileLink(str(OUT), result_html_prefix="Download / open the dashboard: "))

Saved C:\Users\alaa.rashwan\Documents\1-Teaching\GitHub Repos\ISE518\10_Descriptive_analytics\power_faults_dashboard.html  (4.74 MB)


c:\Users\alaa.rashwan\Documents\1-Teaching\GitHub Repos\ISE518\10_Descriptive_analytics\power_faults_dashboard.html

In [12]:
# Inline preview (some notebook front-ends block iframes to local files — if so, just open the HTML
# file in your browser via the link above).
IFrame(src=str(OUT), width="100%", height=900)

## 9. Iterate (optional)

Want changes? Send Claude the current HTML plus your feedback — it edits instead of starting over.
The placeholder trick is reused, so the data is not re-sent.

In [13]:
# FEEDBACK = ""   # e.g. "Use KFUPM green (#027E40) as the accent and add a downtime histogram."

# if FEEDBACK:
#     template = m.group(1).strip()   # Claude's original HTML, still containing the placeholder
#     with client.beta.messages.stream(**{**request, "messages": [
#         {"role": "user", "content": USER_PROMPT},
#         {"role": "assistant", "content": f"```html\n{template}\n```"},
#         {"role": "user", "content": f"Revise the dashboard: {FEEDBACK}\nKeep `const RAW_DATA = {DATA_PLACEHOLDER};` "
#                                     "and return the full updated HTML in one ```html block."},
#     ]}) as stream:
#         response = stream.get_final_message()
#     answer = "".join(b.text for b in response.content if b.type == "text")
#     print("stop_reason:", response.stop_reason, "→ re-run sections 7 and 8 to rebuild the file.")
# else:
#     print("No feedback set — skipping.")